# Exercise 00 - PyTorch Primer

**CAS Deep Learning — Computer Vision**

## Learning Objectives

After this notebook you should be able to:
- Load a standard image dataset with torchvision
- Inspect tensor shapes, value ranges, and batch structure
- Define a simple multilayer perceptron in PyTorch
- Run a minimal raw-PyTorch training and evaluation loop

**Fill in code where you see `# YOUR CODE HERE`.**

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    # No extra packages needed for this notebook on Colab.
    print("Mounting Drive")
    from google.colab import drive

    ROOT_PATH = Path("/content/drive")
    drive.mount(str(ROOT_PATH))
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

## Imports

We keep the dependency footprint deliberately small: only PyTorch, torchvision, matplotlib, and tqdm.

In [ ]:
import torch
import torchvision
from matplotlib import pyplot as plt
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from tqdm.auto import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Part 1 - Load and inspect a dataset

We load and inspect a dataset using torchvision dataset classes.

In [ ]:
ds_train_raw = torchvision.datasets.MNIST(root=DATA_PATH, train=True, download=True)
ds_test_raw = torchvision.datasets.MNIST(root=DATA_PATH, train=False, download=True)

pil_image, label = ds_train_raw[0]
tensor_image = transforms.ToTensor()(pil_image)

print(f"Sample type: {type(pil_image)} | label: {label}")
print(
    f"Tensor shape: {tensor_image.shape} | dtype: {tensor_image.dtype} | min/max: ({tensor_image.min().item():.1f}, {tensor_image.max().item():.1f})"
)

fig, ax = plt.subplots(figsize=(3, 3))
_ = ax.imshow(pil_image, cmap="gray")
_ = ax.set_title(f"MNIST label: {label}")
_ = ax.axis("off")
plt.show()

In [ ]:
assert len(ds_train_raw) == 60000
assert len(ds_test_raw) == 10000
assert tensor_image.shape == (1, 28, 28)
assert tensor_image.min().item() >= 0.0
assert tensor_image.max().item() <= 1.0
print("OK")

## Part 2 - Build a tensor pipeline with DataLoader

A model trains on mini-batches of tensors, not on single PIL images. We therefore add a transform and build small train/test subsets so this notebook stays fast on CPU.

In [ ]:
transform = transforms.ToTensor()

ds_train = torchvision.datasets.MNIST(
    root=DATA_PATH,
    train=True,
    download=False,
    transform=transform,
)
ds_test = torchvision.datasets.MNIST(
    root=DATA_PATH,
    train=False,
    download=False,
    transform=transform,
)

train_subset = Subset(ds_train, range(4096))
test_subset = Subset(ds_test, range(1024))

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

In [ ]:
assert images.shape == (128, 1, 28, 28)
assert labels.shape == (128,)
assert images.dtype == torch.float32
print("OK")

## Part 3 - Define a first PyTorch model

For preparation we use a small MLP. It is not the right architecture for real image problems, but it exposes the PyTorch mechanics very clearly.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size: int = 128, num_classes: int = 10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = MLP().to(device)
logits = model(images.to(device))
print(logits.shape)

In [ ]:
assert logits.shape == (128, 10)
print("OK")

## Part 4 - Train and evaluate

The course exercises use raw PyTorch loops rather than a high-level trainer. This notebook keeps that philosophy, but on a deliberately small subset.

In [ ]:
def train_one_epoch(model, data_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch_images, batch_labels in tqdm(data_loader, leave=False):
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        logits = model(batch_images)
        loss = loss_fn(logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_images.size(0)
        total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total_examples += batch_images.size(0)

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for batch_images, batch_labels in data_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_images)
            loss = loss_fn(logits, batch_labels)

            total_loss += loss.item() * batch_images.size(0)
            total_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total_examples += batch_images.size(0)

    return total_loss / total_examples, total_correct / total_examples

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, loss_fn, device
    )
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        }
    )
    print(
        f"Epoch {epoch + 1}: train_acc={train_acc:.3f} | test_acc={test_acc:.3f} | train_loss={train_loss:.3f}"
    )

In [ ]:
final_metrics = history[-1]
assert final_metrics["test_acc"] > 0.85, final_metrics
print("OK")

## Summary

This preparatory notebook covers the main PyTorch building blocks used throughout the course: dataset, transform, dataloader, model, loss, optimizer, training loop, and evaluation loop.

In the main course notebooks, the data becomes more realistic and the models become more image-specific, but the overall training pattern stays the same.